In [1]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score

In [2]:
X, y = load_digits(return_X_y=True)
print("Dataset shape:", X.shape)


Dataset shape: (1797, 64)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)


In [4]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [5]:
models = {
    "SVM": SVC(),
    "Random Forest": RandomForestClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
}

In [6]:
grids = {
    "SVM": {"C": [0.1, 1, 10], "gamma": ["scale", 0.01, 0.001], "kernel": ["linear", "rbf"]},
    "Random Forest": {"n_estimators": [50, 100, 200], "max_depth": [None, 5, 10]},
    "AdaBoost": {"n_estimators": [50, 100, 200], "learning_rate": [0.1, 0.5, 1.0],
                 "estimator": [DecisionTreeClassifier(max_depth=d) for d in (1, 2, 3)]},
}

In [7]:
print("\n BEFORE OPTIMIZATION ")
before = {}
for name, model in models.items():
    cv_score = cross_val_score(model, X_train, y_train, cv=5).mean()
    model.fit(X_train, y_train)
    test_acc = accuracy_score(y_test, model.predict(X_test))
    before[name] = (cv_score, test_acc)
    print(f"{name:14s} CV accuracy = {cv_score:.4f}   Test accuracy = {test_acc:.4f}")


 BEFORE OPTIMIZATION 
SVM            CV accuracy = 0.9837   Test accuracy = 0.9800
Random Forest  CV accuracy = 0.9748   Test accuracy = 0.9622
AdaBoost       CV accuracy = 0.7609   Test accuracy = 0.7467


In [ ]:
print("\n AFTER OPTIMIZATION  ")
after = {}
for name, model in models.items():
    grid = GridSearchCV(model, grids[name], cv=5)
    grid.fit(X_train, y_train)
    test_acc = accuracy_score(y_test, grid.predict(X_test))
    after[name] = (grid.best_score_, test_acc)
    best = grid.best_params_.copy()
    if "estimator" in best:
        best["estimator"] = "DecisionTree(max_depth=%d)" % best["estimator"].max_depth
    print(f"{name:14s} CV accuracy = {grid.best_score_:.4f}   Test accuracy = {test_acc:.4f}")
    print("   Best parameters:", best)



 AFTER OPTIMIZATION  
SVM            CV accuracy = 0.9859   Test accuracy = 0.9800
   Best parameters: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Random Forest  CV accuracy = 0.9777   Test accuracy = 0.9667
   Best parameters: {'max_depth': None, 'n_estimators': 200}


In [ ]:
print("\n COMPARISON  ")
print(f"{'Model':14s} {'Before':>8s} {'After':>8s} {'Gain':>8s}")
for name in models:
    b, a = before[name][1], after[name][1]
    print(f"{name:14s} {b:8.4f} {a:8.4f} {a - b:+8.4f}")